# QTracker Multi-Track Training — Google Colab

Trains the joint denoising + segmentation model from `.npz` files (no ROOT required).

**Setup steps:**
1. Upload the `.npz` files to Google Drive (produced by `data/root_to_numpy.py` on Rivanna)
2. Connect to a Colab GPU runtime (A100 recommended via Colab Pro)
3. Run cells top to bottom

**Data expected in Google Drive:**
```
MyDrive/qtracker_data/
    train_low.npz
    train_med.npz
    train_high.npz
    val.npz
```

In [ ]:
# ── 1. Install dependencies (Colab only) ──────────────────────────────────────
import subprocess, sys

def pip_install(*pkgs):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *pkgs])

pip_install("mlflow", "tensorflow")
print("Dependencies ready.")

In [ ]:
# ── 2. Mount Google Drive & set data paths ─────────────────────────────────────
from google.colab import drive
drive.mount("/content/drive")

DATA_DIR = "/content/drive/MyDrive/qtracker_data"
CKPT_DIR = "/content/drive/MyDrive/qtracker_checkpoints"
MLRUNS_DIR = "/content/drive/MyDrive/qtracker_mlruns"

import os
os.makedirs(CKPT_DIR, exist_ok=True)
os.makedirs(MLRUNS_DIR, exist_ok=True)
print("Drive mounted. Data dir:", DATA_DIR)

In [ ]:
# ── 3. Clone repo & add models/ to path ───────────────────────────────────────
import subprocess
repo_dir = "/content/Qtracker_basic"
if not os.path.exists(repo_dir):
    subprocess.check_call([
        "git", "clone", "--branch", "AnveshTrackFinderWork", "--depth", "1",
        "https://github.com/uva-spin/Qtracker_basic.git", repo_dir
    ])
else:
    subprocess.check_call(["git", "-C", repo_dir, "pull"])

sys.path.insert(0, os.path.join(repo_dir, "QTracker_training", "models"))
print("Repo ready. sys.path updated.")

In [ ]:
# ── 4. GPU check ──────────────────────────────────────────────────────────────
import tensorflow as tf
gpus = tf.config.list_physical_devices("GPU")
if not gpus:
    raise RuntimeError("No GPU detected — go to Runtime > Change runtime type > GPU")
print(f"TF {tf.__version__} | GPUs: {[g.name for g in gpus]}")

In [ ]:
# ── 5. Load data via numpy loader (no ROOT needed) ───────────────────────────
from data_loader_numpy import load_data_numpy
import numpy as np

MAX_PAIRS = 5

X_low, X_clean_low, y_mup_low, y_mum_low = load_data_numpy(
    f"{DATA_DIR}/train_low.npz", max_pairs=MAX_PAIRS
)
X_val, X_clean_val, y_mup_val, y_mum_val = load_data_numpy(
    f"{DATA_DIR}/val.npz", max_pairs=MAX_PAIRS
)

# Stack mup/mum into (N, max_pairs, 2, 62) for the model
y_low = np.stack([y_mup_low, y_mum_low], axis=2)
y_val = np.stack([y_mup_val, y_mum_val], axis=2)

print("Low train:", X_low.shape, "Val:", X_val.shape)

In [ ]:
# ── 6. Build model ────────────────────────────────────────────────────────────
from backbones import unetpp_backbone
from layers import AxialAttention
import tensorflow as tf
from tensorflow.keras import layers

strategy = tf.distribute.MirroredStrategy()
print(f"Devices: {strategy.num_replicas_in_sync}")

NUM_DETECTORS = 62
NUM_ELEMENTS = 201
DENOISE_BASE = 32
BASE = 64
DROPOUT_BN = 0.5
DROPOUT_ENC = 0.4
DROPOUT_ATTN = 0.1
USE_ATTN = True
BATCH_NORM = True
MAX_PAIRS = 5

with strategy.scope():
    inp = tf.keras.Input(shape=(NUM_DETECTORS, NUM_ELEMENTS, 1))
    padded = layers.ZeroPadding2D(((1, 1), (4, 3)))(inp)

    # Denoiser backbone
    denoise_out, _ = unetpp_backbone(
        padded, base_filters=DENOISE_BASE,
        dropout_rate_bn=DROPOUT_BN, dropout_rate_enc=DROPOUT_ENC,
        batch_norm=BATCH_NORM, use_attn=False, use_attn_ffn=False,
        name_prefix="denoise"
    )

    # Segmentation backbone
    seg_features, _ = unetpp_backbone(
        denoise_out, base_filters=BASE,
        dropout_rate_bn=DROPOUT_BN, dropout_rate_enc=DROPOUT_ENC,
        batch_norm=BATCH_NORM, use_attn=USE_ATTN, use_attn_ffn=False,
        name_prefix="seg"
    )

    cropped = layers.Cropping2D(((1, 1), (4, 3)))(seg_features)

    if USE_ATTN:
        cropped = AxialAttention(BASE, DROPOUT_ATTN)(cropped)
        cropped = AxialAttention(BASE, DROPOUT_ATTN)(cropped)

    logits = layers.Conv2D(MAX_PAIRS * 2, 1, padding="same")(cropped)
    permuted = layers.Permute((3, 1, 2))(logits)
    reshaped = layers.Reshape((MAX_PAIRS, 2, NUM_DETECTORS, NUM_ELEMENTS))(permuted)
    seg_output = layers.Softmax(name="segment")(reshaped)

    model = tf.keras.Model(inputs=inp, outputs=[denoise_out, seg_output])

model.summary(line_length=100)

In [ ]:
# ── 7. Compile & train (low complexity phase) ─────────────────────────────────
import mlflow

os.environ["MLFLOW_TRACKING_URI"] = f"file://{MLRUNS_DIR}"
mlflow.set_experiment("colab_multi_track")

from losses import combined_loss, denoise_loss

LR = 3e-4
POS_WEIGHT = 20.0
BATCH_SIZE = 64
EPOCHS_LOW = 30

with strategy.scope():
    model.compile(
        optimizer=tf.keras.optimizers.Adam(LR),
        loss={
            "segment": combined_loss(pos_weight=POS_WEIGHT),
            "functional": denoise_loss(),
        },
        loss_weights={"segment": 1.0, "functional": 0.5},
    )

ckpt_cb = tf.keras.callbacks.ModelCheckpoint(
    filepath=f"{CKPT_DIR}/multi_track_colab.keras",
    save_best_only=True, monitor="val_segment_loss", verbose=1
)
es_cb = tf.keras.callbacks.EarlyStopping(
    monitor="val_segment_loss", patience=10, restore_best_weights=True
)

with mlflow.start_run(run_name="colab_low"):
    mlflow.log_params({
        "phase": "low", "epochs": EPOCHS_LOW, "lr": LR,
        "batch_size": BATCH_SIZE, "pos_weight": POS_WEIGHT,
        "base": BASE, "denoise_base": DENOISE_BASE, "max_pairs": MAX_PAIRS,
    })
    history = model.fit(
        X_low, {"segment": y_low, "functional": X_clean_low},
        validation_data=(X_val, {"segment": y_val, "functional": X_clean_val}),
        epochs=EPOCHS_LOW, batch_size=BATCH_SIZE,
        callbacks=[ckpt_cb, es_cb],
    )
    for epoch, (tl, vl) in enumerate(zip(
        history.history["segment_loss"], history.history["val_segment_loss"]
    )):
        mlflow.log_metrics({"train_seg_loss": tl, "val_seg_loss": vl}, step=epoch)
print("Low phase done.")

In [ ]:
# ── 8. Continue training on med & high complexity ─────────────────────────────
for phase, npz_name, lr, epochs in [
    ("med",  "train_med.npz",  1e-4,  30),
    ("high", "train_high.npz", 3e-5, 60),
]:
    X_phase, X_clean_phase, y_mup_phase, y_mum_phase = load_data_numpy(
        f"{DATA_DIR}/{npz_name}", max_pairs=MAX_PAIRS
    )
    y_phase = np.stack([y_mup_phase, y_mum_phase], axis=2)

    with strategy.scope():
        model.optimizer.learning_rate.assign(lr)

    with mlflow.start_run(run_name=f"colab_{phase}"):
        mlflow.log_params({"phase": phase, "epochs": epochs, "lr": lr})
        history = model.fit(
            X_phase, {"segment": y_phase, "functional": X_clean_phase},
            validation_data=(X_val, {"segment": y_val, "functional": X_clean_val}),
            epochs=epochs, batch_size=BATCH_SIZE,
            callbacks=[ckpt_cb, es_cb],
        )
        for epoch, (tl, vl) in enumerate(zip(
            history.history["segment_loss"], history.history["val_segment_loss"]
        )):
            mlflow.log_metrics({"train_seg_loss": tl, "val_seg_loss": vl}, step=epoch)
    print(f"{phase} phase done.")

In [ ]:
# ── 9. Save final model & sync mlruns summary ─────────────────────────────────
model.save(f"{CKPT_DIR}/multi_track_colab_final.keras")
print("Model saved to Drive.")
print(f"MLflow runs stored in: {MLRUNS_DIR}")
print("To view locally: rsync mlruns to your Mac, then run `mlflow ui --backend-store-uri ./mlruns`")